# mtlab — Example Usage

This notebook demonstrates the main functions in `mtlab.py` using the provided test data.

| Function | Description |
|---|---|
| `mtlab.plot()` | Publication-style line plot |
| `mtlab.moving_average()` | Smoothing filter |
| `mtlab.diff_fitting()` | Fit derivative of Lorentzian (ST-FMR) |
| `mtlab.residual()` / `mtlab.fit_data()` | Fit symmetric + antisymmetric Lorentzian |

> **Colab users:** run the first setup cell to clone the repo.

## 0. Setup (Colab only)

In [ ]:
# Run this cell only on Google Colab
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/ttt50966/mtlab.git
    %cd mtlab
    !pip install lmfit -q

## 1. Import

In [ ]:
import mtlab
import pandas as pd
import numpy as np
import lmfit
import matplotlib.pyplot as plt
%matplotlib inline

print('mtlab loaded successfully')

## 2. `mtlab.plot` — Multi-line plot

`plot_test.csv` contains two (x, y) pairs.

In [ ]:
df_plot = pd.read_csv('test_data/plot_test.csv')
df_plot

In [ ]:
line = [
    {'x': df_plot['x1'].values, 'y': df_plot['y1'].values, 'label': 'Dataset 1'},
    {'x': df_plot['x2'].values, 'y': df_plot['y2'].values, 'label': 'Dataset 2'},
]

mtlab.plot(line, dpiValue=150, title='Plot Example', xlabel='x', ylabel='y', saveFig=False)

## 3. `mtlab.moving_average` — Smoothing

Compare raw signal vs. smoothed versions with different window sizes.

In [ ]:
df_fit = pd.read_csv('test_data/fitting.csv')
df_fit.head()

In [ ]:
x_raw = df_fit['x'].values
y_raw = df_fit['y'].values

smooth_lines = [{'x': x_raw, 'y': y_raw, 'label': 'raw'}]

for w in [3, 5, 7]:
    y_sm = mtlab.moving_average(y_raw, w)
    smooth_lines.append({'x': x_raw, 'y': y_sm, 'label': f'window={w}'})

mtlab.plot(smooth_lines, dpiValue=150, title='Moving Average', xlabel='H (Oe)', ylabel='Signal (V)', saveFig=False)

## 4. `mtlab.diff_fitting` — Derivative Lorentzian fit

The ST-FMR signal is modelled as the derivative of a Lorentzian:

$$f(H) = \frac{A T^2 (2H_0 - 2H)}{(T^2 + (H_0 - H)^2)^2} - \frac{2BT(H_0-H)(2H_0-2H)}{(T^2+(H_0-H)^2)^2} - \frac{2BT}{T^2+(H_0-H)^2}$$

where $T$ is linewidth, $H_0$ is resonance field, $A$ (symmetric) and $B$ (antisymmetric) are amplitudes.

In [ ]:
x = df_fit['x'].values
y = df_fit['y'].values

# Set initial guesses
params = lmfit.Parameters()
params.add('A', value=-0.08)
params.add('B', value=0.1)
params.add('T', value=5)
params.add('H', value=800)

out = mtlab.diff_fitting(params, x, y)

print(lmfit.fit_report(out))

In [ ]:
x_dense = np.linspace(x.min(), x.max(), 500)
y_fit = mtlab.diff_fit_data(out.params, x_dense)

result_lines = [
    {'x': x, 'y': y, 'label': 'data'},
    {'x': x_dense, 'y': y_fit, 'label': 'diff_fitting'},
]

mtlab.plot(result_lines, dpiValue=150, title='Diff Fitting', xlabel='H (Oe)', ylabel='dV/dH (a.u.)', saveFig=False)

# Print fitted parameters
print(f"H_res = {out.params['H'].value:.2f} Oe")
print(f"Linewidth T = {out.params['T'].value:.2f} Oe")
print(f"A = {out.params['A'].value:.4f}")
print(f"B = {out.params['B'].value:.4f}")

## 5. `mtlab.residual` / `mtlab.fit_data` — Symmetric + Antisymmetric Lorentzian

Alternative decomposition fitting the signal directly (not the derivative):

$$V(H) = \frac{S \cdot T^2}{(H - H_{\rm FMR})^2 + T^2} + \frac{A \cdot T(H - H_{\rm FMR})}{(H - H_{\rm FMR})^2 + T^2} + c$$

In [ ]:
params2 = lmfit.Parameters()
params2.add('S',    value=0.05)
params2.add('A',    value=0.01)
params2.add('T',    value=5)
params2.add('c',    value=0)
params2.add('Hfmr', value=802)

out2 = lmfit.minimize(mtlab.residual, params2, args=(x, y))
print(lmfit.fit_report(out2))

In [ ]:
y_fit2 = mtlab.fit_data(out2.params, x_dense)

result2_lines = [
    {'x': x, 'y': y, 'label': 'data'},
    {'x': x_dense, 'y': y_fit2, 'label': 'sym+antisym fit'},
]

mtlab.plot(result2_lines, dpiValue=150, title='Lorentzian Fitting', xlabel='H (Oe)', ylabel='V (a.u.)', saveFig=False)

print(f"H_FMR = {out2.params['Hfmr'].value:.2f} Oe")
print(f"Linewidth T = {out2.params['T'].value:.2f} Oe")